# Application de PCA au dataset Iris

Le dataset Iris est un classique pour apprendre à utiliser les méthodes de Machine Learning. Nous allons ici nous en servir pour voir comment mettre en place PCA et quels sont les pièges à éviter.

## Import des données

Dans ses versions > 0.23, sklearn permet de récupérer le dataset iris directement sous forme de DataFrame.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy
import plotly.express as px
import scipy.stats
import sklearn.datasets
import sklearn.decomposition
import sklearn.preprocessing


df = px.data.iris()

print("5 premières lignes")
print(df.head(5))
print()
print("Description des features numériques")
print(df.describe())

## Préparation des features et targets

Les targets nous serviront ici pour la visualisation seulement : pas pour l'apprentissage.

In [ ]:
X = df.drop(columns=["species", "species_id"]).values
y = df["species_id"].values

## Visualisation des données

In [ ]:
styling = dict(template="seaborn", width=800, height=600)


def show(fig):
    fig.update_traces(
        marker=dict(line=dict(width=1, color="DarkSlateGrey")),
        selector=dict(mode="markers"),
    )
    fig.show()


fig = px.scatter_3d(
    df,
    x="sepal_length",
    y="sepal_width",
    z="petal_length",
    color="species",
    hover_data=[
        "petal_width"
    ],  # permet d'afficher la valeur de "petal_width" en overlay
    **styling,
)
show(fig)

## Calcul de PCA

In [ ]:
pca = sklearn.decomposition.PCA(n_components=3)
pca.fit(X)

X_pca = pca.transform(X)

print("variance expliquée :", pca.explained_variance_ratio_)
print("variance expliquée cumulée :", numpy.cumsum(pca.explained_variance_ratio_))

## Visualisation des projections linéaires apprises

In [ ]:
fig = px.scatter_3d(X_pca, x=0, y=1, z=2, color=df["species"], **styling)
show(fig)

## Réduction de dimensionnalité

In [ ]:
fig = px.scatter(X_pca, x=0, y=1, color=df["species"], **styling)
show(fig)

## Exploitation des composants principaux

In [ ]:
print("Composants", pca.components_)

for k, v in enumerate(numpy.argsort(numpy.abs(pca.components_))):
    print()
    print(f"Coefficient des features pour la composante {k}")
    for i in v[::-1]:
        print(f"  {df.columns[i]}: {pca.components_[k][i]}")

# Unités des features et variance

Ces résultats sont à prendre avec du recul : on pourrait croire que 92% de la variance est conservée grâce à la projection du premier vecteur propre . Cependant, il n'y a pas eu de normalisation des données, donc une feature avec des valeurs plus grandes dominera les autres dans la conservation de la variance.

Il y a une solution : la normalisation des features.

In [ ]:
X_scaler = sklearn.preprocessing.StandardScaler()
X_scaler.fit(X)
X_scaled = X_scaler.transform(X)
# X_scaled = X_scaler.fit_transform(X)
print(X_scaled.shape)

In [ ]:
pca = sklearn.decomposition.PCA(n_components=3)
pca.fit(X_scaled)
X_scaled_pca = pca.transform(X_scaled)

print(
    "variance expliquée pour chaque composante :",
    ", ".join(f"{v:.2f}" for v in pca.explained_variance_ratio_),
)
print(
    "variance expliquée cumulée :",
    ", ".join(f"{v:.2f}" for v in numpy.cumsum(pca.explained_variance_ratio_)),
)

In [ ]:
fig = px.scatter_3d(X_scaled_pca, x=0, y=1, z=2, color=df["species"], **styling)
show(fig)

In [ ]:
print("Composants", pca.components_)

for k, v in enumerate(numpy.argsort(numpy.abs(pca.components_))):
    print()
    print(f"Coefficient des features pour la composante {k}")
    for i in v[::-1]:
        print(f"  {df.columns[i]}: {pca.components_[k][i]}")

In [ ]:
t = numpy.linspace(0, numpy.pi * 2, 100)

plt.figure(figsize=(10, 10))
plt.xlim((-1.1, 1.1))
plt.ylim((-1.1, 1.1))
for feature in range(X_scaled.shape[1]):
    corr_0 = scipy.stats.pearsonr(X_scaled[:, feature], X_scaled_pca[:, 0])[0]
    corr_1 = scipy.stats.pearsonr(X_scaled[:, feature], X_scaled_pca[:, 1])[0]
    r = math.atan2(corr_1, corr_0)
    plt.arrow(0, 0, corr_0, corr_1, length_includes_head=True, head_width=0.03)
    plt.text(
        corr_0 / 2 + (math.cos(r + 0.1) - math.cos(r)) / 2,
        corr_1 / 2 + (math.sin(r + 0.1) - math.sin(r)) / 2,
        df.columns[feature],
        rotation=numpy.degrees(r),
        size=16,
        ha="center",
        va="center",
    )
plt.xlabel("Première composante")
plt.ylabel("Deuxième composante")
plt.plot(numpy.cos(t), numpy.sin(t))
plt.show()